# Mamba + Workspace POC — Kaggle T4

Run cells B and D (the go/no-go pair) on a Kaggle T4 GPU.

**Setup before running:**
1. Settings → Accelerator → **GPU T4 x2** (single T4 is fine)
2. Settings → Internet → **On** (needed for pip install + git clone)
3. Add a Kaggle Secret called `github_token` with a GitHub Personal Access Token
   that has `repo` scope (Settings → Secrets → Add Secret)

The notebook clones your private GitHub repo, uses the pure-PyTorch Mamba2
fallback (no `mamba-ssm` needed), and enables fp16 AMP for T4.

In [ ]:
# Install dependencies — pure-PyTorch path, no mamba-ssm needed
!pip install -q einops pyyaml wandb numpy

In [ ]:
import os
import subprocess

# Clone the private repo using a GitHub token stored in Kaggle Secrets
# Before running: add a Kaggle Secret named "github_token" with a GitHub PAT (repo scope)
# Settings → Secrets → Add Secret → Name: github_token, Value: <your PAT>

work_dir = '/kaggle/working/mamba-poc'
repo_url = 'https://github.com/rbf22/jasper.git'

if os.path.exists(work_dir):
    print(f"Working dir already exists, pulling latest...")
    os.chdir(work_dir)
    subprocess.run(['git', 'pull'], check=True)
else:
    # Try with Kaggle Secrets token
    try:
        import kaggle_secrets
        user_secrets = kaggle_secrets.UserSecretsClient()
        gh_token = user_secrets.get_secret("github_token")
        # Inject token into URL: https://<token>@github.com/...
        authed_url = repo_url.replace('https://', f'https://{gh_token}@')
        print(f"Cloning private repo with token auth...")
        subprocess.run(['git', 'clone', authed_url, '/kaggle/working/jasper'], check=True)
        # Move mamba-poc to the expected location
        import shutil
        shutil.move('/kaggle/working/jasper/mamba-poc', work_dir)
        os.chdir(work_dir)
        print(f"Cloned to: {work_dir}")
    except Exception as e:
        print(f"Kaggle Secrets auth failed: {e}")
        print("Falling back to public clone (will fail if repo is private)...")
        subprocess.run(['git', 'clone', repo_url, '/kaggle/working/jasper'], check=True)
        import shutil
        shutil.move('/kaggle/working/jasper/mamba-poc', work_dir)
        os.chdir(work_dir)

print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir('.'))

In [ ]:
# Verify GPU and environment
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Torch: {torch.__version__}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# T4 is Turing — no bf16, but fp16 works great with AMP
# train.py auto-detects CUDA and enables fp16 autocast + GradScaler

In [ ]:
# Quick data test — verifies all 3 task generators
!python data.py

In [ ]:
# Param count check — verify all 4 cells are ~30M
!python model.py

In [ ]:
# Smoke test — 5M params, Task 1 depth-4, ~200 steps (~5 min on T4)
# Confirms loss drops and model generates verifiable answers
!python train.py --config configs/cell_d_kaggle.yaml --smoke-test

## Cell B — Hybrid Baseline (12 Mamba2 + 2 Attention)

This is the real baseline the workspace must beat. ~12 hours on T4 for 10K steps.

AMP (fp16) is enabled via the kaggle config. Checkpoints save every 15 min and auto-resume if the session disconnects.

In [ ]:
# Optional: Set wandb API key via Kaggle Secrets
# import kaggle_secrets
# user_secrets = kaggle_secrets.UserSecretsClient()
# wandb_key = user_secrets.get_secret("wandb_api_key")
# os.environ['WANDB_API_KEY'] = wandb_key

# To enable wandb, add `wandb: true` to the config or edit the yaml below:
# with open('configs/cell_b_kaggle.yaml') as f:
#     cfg = yaml.safe_load(f)
# cfg['wandb'] = True
# cfg['run_name'] = 'cellB-kaggle-seed1'
# with open('configs/cell_b_kaggle.yaml', 'w') as f:
#     yaml.dump(cfg, f)

# Train Cell B — full run with AMP
!python train.py --config configs/cell_b_kaggle.yaml

## Cell D — Full Architecture (Hybrid + Workspace + Recurrent Core)

The cell that decides go/no-go. ~12 hours on T4 for 10K steps.

If Cell B's run is still in `checkpoints/`, it won't be overwritten — Cell D saves to `cellD_latest.pt`.

In [ ]:
# Train Cell D — full run with AMP
!python train.py --config configs/cell_d_kaggle.yaml

## Analysis (R2, R3, R4)

In [ ]:
# Run all analyses on Cell D
# R2: K sweep (accuracy vs inference K at each depth)
# R3: Linear probes (decodability of workspace slots)
# R4: Selective ablation (J-space signature)
!python probe.py --checkpoint checkpoints/cellD_latest.pt --config configs/cell_d_kaggle.yaml --all

In [ ]:
# Save checkpoints to Kaggle output — survives session end
# After the session ends, you can download these from the Output tab
import shutil
import os

# Copy checkpoints
if os.path.exists('checkpoints'):
    for f in os.listdir('checkpoints'):
        src = os.path.join('checkpoints', f)
        dst = os.path.join('/kaggle/working', f)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"Saved: {dst} ({os.path.getsize(src) / 1e6:.1f} MB)")

# Also save any probe output files
for fname in os.listdir('.'):
    if fname.startswith('probe_') and (fname.endswith('.json') or fname.endswith('.csv') or fname.endswith('.png')):
        shutil.copy2(fname, os.path.join('/kaggle/working', fname))
        print(f"Saved: {fname}")

print("\nAll outputs saved to /kaggle/working/ — download from the Output tab after session ends.")